# Retrospective ecotype imputation with calibrated abstention

This notebook reads the canonical preprocessed sightings table, trains a day-resolution retrospective model, evaluates it with two cross-fitting strategies, assigns only predictions that pass the full reject policy, and creates an animated 2025 map.

**Display classes**

- `SRKW`
- `TRANSIENT`
- `SRKW_ASSIGNED`
- `TRANSIENT_ASSIGNED`
- `UNKNOWN_STILL`

The model treats canonical noon UTC as a **date label**, not a precise encounter time. Same-day evidence is day lag 0; retrospective context also includes earlier and later dates.

## 1. Load the local source

This adds the repository's production `src` directory to Python's import path. No package installation is required.

In [ ]:
import os
from pathlib import Path
REPO_ROOT = Path(os.environ["MARINE_MAMMALS_WORKSPACE_ROOT"]).expanduser().resolve()
# Use the installed toolkit. This root locates existing data, not Python source.
print(f"Data workspace: {REPO_ROOT}")


## 2. Paths and model configuration

In [ ]:
from dataclasses import replace
import json
import pandas as pd
from IPython.display import display
from marine_mammal_toolkit.tools.observations.impute.settings import load_imputation_settings
from marine_mammal_toolkit.cetaceans.killer_whales.resources import config_path
from marine_mammal_toolkit.tools._core.config import workspace

from marine_mammal_toolkit.tools.observations.impute import FeatureConfig
from marine_mammal_toolkit.tools.observations.impute import ImputationConfig
from marine_mammal_toolkit.tools.observations.impute import ModelConfig
from marine_mammal_toolkit.tools.observations.impute import load_preprocessed_sightings
from marine_mammal_toolkit.tools.observations.impute import make_animated_map
from marine_mammal_toolkit.tools.observations.impute import plot_confusion
from marine_mammal_toolkit.tools.observations.impute import plot_reliability
from marine_mammal_toolkit.tools.observations.impute import plot_risk_coverage
from marine_mammal_toolkit.tools.observations.impute import run_imputation_workflow

# These are the source-neutral outputs of `data process sightings`. The imputer
# consumes them before the optional grid-count stage.
OBSERVATIONS_PATH = REPO_ROOT / "data/processed/domain/whale_layer/sightings/observations.parquet"
ASSOCIATIONS_PATH = REPO_ROOT / "data/processed/domain/whale_layer/sightings/associations.parquet"
if not OBSERVATIONS_PATH.exists():
    raise FileNotFoundError(
        f"Normalized observations not found at {OBSERVATIONS_PATH}. "
        "Run the sightings collection and processing stages first."
    )
if not ASSOCIATIONS_PATH.exists():
    ASSOCIATIONS_PATH = None

OUTPUT_DIR = REPO_ROOT / "outputs/ecotype_imputation_retrospective"
# Consume an existing, checksum-verified seascape water graph. Do not build it here.
with workspace(REPO_ROOT):
    settings = load_imputation_settings(os.environ.get("MARINE_MAMMALS_CONFIG", config_path()))
WATER_H3_GRID_PATH = Path(os.environ["MARINE_MAMMALS_WATER_GRID"])
MODEL_RADIUS_KM = 40.0
feature_config = FeatureConfig(
    max_radius_km=MODEL_RADIUS_KM,
    distance_scale_km=10.0,
    time_scale_days=4.0,
    max_day_lag=14,
    support_unit_km=2.5,
    query_chunk_size=5_000,
    water_network_config_path=settings.config.feature.water_network_config_path,
    marine_h3_resolution=settings.config.feature.marine_h3_resolution,
    marine_fallback_to_haversine=False,
    max_target_snap_km=5.0,
    strong_local_support_floor=0.01,
    temporal_windows=(
        ("same_day", 0, 0),
        ("past_1d", -1, -1),
        ("past_2_3d", -3, -2),
        ("past_4_7d", -7, -4),
        ("past_8_14d", -14, -8),
        ("future_1d", 1, 1),
        ("future_2_3d", 2, 3),
        ("future_4_7d", 4, 7),
        ("future_8_14d", 8, 14),
    ),
)

model_config = ModelConfig(
    model_kind="extra_trees",
    n_estimators=150,
    n_splits=3,  # More independent transient encounters in each outer training fold.
    calibration_method="sigmoid",  # More stable when transferred to the full-data fit.
    conformal_alpha=0.10,
    target_selective_error=0.05,
    risk_confidence=0.95,
    nested_decision_splits=2,  # Reserve half of each outer training fold for decisions.
    max_evaluation_per_class=5_000,
    ood_n_estimators=75,
    max_training_majority_ratio=10.0,
    encounter_radius_km=3.0,
    encounter_association_radius_km=20.0,
    encounter_timestamp_tolerance_hours=6.0,
    use_regime_experts=True,
    min_expert_examples=100,
    enable_hyperparameter_tuning=True,
    tuning_max_candidates=8,
    tuning_n_jobs=2,
    enable_prediction_stability=True,
    maximum_stability_probability_shift=0.20,
    final_calibration_strategy="encounter",
)

config = ImputationConfig(
    regime="retrospective",
    feature=feature_config,
    model=model_config,
    map_year=2025,
    map_frame="week",
)

# Practical validation set: reconstruction, encounter-held-out, and blocked.
# Add "purged_blocked" only for the expensive context-free stress test.
EVALUATION_STRATEGIES = ("reconstruction", "encounter", "blocked")

OBSERVATIONS_PATH, ASSOCIATIONS_PATH, OUTPUT_DIR

## 3. Inspect the preprocessed inputs

Known `NRKW` and `OFFSHORE` records are not binary training labels. They supply known-OTHER context that can veto an unsafe SRKW/transient assignment. This cell verifies the normalized sightings-pipeline contract and label populations before starting the expensive workflow.

In [ ]:
observations, associations = load_preprocessed_sightings(
    OBSERVATIONS_PATH,
    ASSOCIATIONS_PATH,
)

required_columns = {
    "OBSERVATION_ID",
    "SIGHTING_DATE",
    "LATITUDE",
    "LONGITUDE",
    "ECOTYPE_DETAIL",
}
missing_columns = required_columns - set(observations.columns)
assert not missing_columns, f"Missing model columns: {sorted(missing_columns)}"
assert observations["OBSERVATION_ID"].is_unique
assert observations["ECOTYPE_DETAIL"].isin(
    ["SRKW", "TRANSIENT", "NRKW", "OFFSHORE", "UNKNOWN", "MIXED"]
).all(), "Unexpected ECOTYPE_DETAIL label found"

summary = (
    observations.assign(YEAR=pd.to_datetime(observations["SIGHTING_DATE"]).dt.year)
    .groupby(["YEAR", "ECOTYPE_DETAIL"], dropna=False)
    .size()
    .rename("N")
    .reset_index()
)

display(summary.tail(30))
display(
    observations["ECOTYPE_DETAIL"]
    .value_counts(dropna=False)
    .rename("N")
    .to_frame()
)
print(f"Normalized observations: {len(observations):,}")
print(f"Date range: {observations['SIGHTING_DATE'].min()} to {observations['SIGHTING_DATE'].max()}")
print(f"Associations loaded: {0 if associations is None else len(associations):,}")

## 4. Fit, cross-validate, calibrate, abstain, and export

This runs two evaluations:

- **Reconstruction:** hides individual known labels while allowing other contemporaneous evidence to remain.
- **Blocked:** holds out date and spatial blocks together as a tougher dependence stress test.

Thresholds are selected from cross-fitted predictions. A label is applied only when the conformal set is a singleton, the case is not flagged as out of distribution, known-OTHER support does not dominate, and the calibrated selective-risk rule passes.

In [ ]:
result = run_imputation_workflow(
        workspace_root=REPO_ROOT,
    observations_path=OBSERVATIONS_PATH,
    associations_path=ASSOCIATIONS_PATH,
    output_dir=OUTPUT_DIR,
    config=config,
    make_map=True,
    evaluate_strategies=EVALUATION_STRATEGIES,
)

imputer = result.imputer
predictions = result.predictions

print(imputer.training_summary_)
display(pd.DataFrame.from_dict(imputer.class_certification_, orient="index"))
if not imputer.tuning_results_.empty:
    display(imputer.tuning_results_)
print(f"Model written to: {result.model_path}")
print(f"Predictions written to: {result.predictions_path}")
print(f"Metrics written to: {result.metrics_path}")
print(f"Map written to: {result.map_path}")

marine_rows = predictions.loc[predictions["DISTANCE_METHOD"].eq("MARINE_LOOKUP")]
display(
    marine_rows[[
        "MARINE_LOOKUP_COVERAGE",
        "MARINE_ROUTED_NEIGHBORS",
        "MARINE_FALLBACK_NEIGHBORS",
        "MARINE_BARRIER_EXCLUDED_NEIGHBORS",
        "MARINE_OUT_OF_RANGE_NEIGHBORS",
    ]].describe().loc[["mean", "max"]]
)

display(
    predictions.groupby("ECOTYPE_LABEL_TIER", dropna=False)
    .agg(
        N=("OBSERVATION_ID", "size"),
        EXPECTED_SRKW=("EXPECTED_SRKW_COUNT", "sum"),
        EXPECTED_TRANSIENT=("EXPECTED_TRANSIENT_COUNT", "sum"),
        EXPECTED_UNKNOWN=("EXPECTED_UNKNOWN_COUNT", "sum"),
    )
)

stability_rows = predictions.loc[predictions["STABILITY_EVALUATED"].fillna(False)]
if not stability_rows.empty:
    display(
        stability_rows.groupby(["PREDICTED_CLASS", "PREDICTION_STABLE"], dropna=False)
        .agg(
            N=("OBSERVATION_ID", "size"),
            MEDIAN_MAX_SHIFT=("MAX_STABILITY_PROBABILITY_SHIFT", "median"),
            MAX_SHIFT=("MAX_STABILITY_PROBABILITY_SHIFT", "max"),
        )
    )

## 5. Headline model metrics

In [ ]:
metric_rows = []
for strategy, evaluation in imputer.evaluations_.items():
    m = evaluation.metrics
    metric_rows.append(
        {
            "strategy": strategy,
            "n": m["N"],
            "brier": m["BRIER"],
            "log_loss": m["LOG_LOSS"],
            "roc_auc": m["ROC_AUC"],
            "ece_10": m["ECE_10"],
            "coverage": m["COVERAGE"],
            "selective_accuracy": m["SELECTIVE_ACCURACY"],
            "selective_error_upper_95": m["SELECTIVE_ERROR_UPPER"],
            "conformal_singleton_rate": m["CONFORMAL_SINGLETON_RATE"],
            "ood_inlier_rate": m["OOD_INLIER_RATE"],
            "encounter_n": m["ENCOUNTER_LEVEL"]["N"],
            "encounter_coverage": m["ENCOUNTER_LEVEL"]["COVERAGE"],
            "encounter_selective_accuracy": m["ENCOUNTER_LEVEL"]["SELECTIVE_ACCURACY"],
            "encounter_error_upper_95": m["ENCOUNTER_LEVEL"]["SELECTIVE_ERROR_UPPER"],
        }
    )

metrics_table = pd.DataFrame(metric_rows).set_index("strategy")
display(metrics_table.style.format("{:.3f}", subset=metrics_table.select_dtypes("number").columns))

class_risk_rows = []
for strategy, evaluation in imputer.evaluations_.items():
    for predicted_class, risk in evaluation.metrics["PREDICTED_CLASS_RISK"].items():
        class_risk_rows.append({"STRATEGY": strategy, "PREDICTED_CLASS": predicted_class, **risk})
display(pd.DataFrame(class_risk_rows))

Interpret both rows together. Reconstruction performance best matches historical attribution. Blocked performance is the colder shower: it limits help from highly related nearby records and reveals how much the model depends on local corroboration.

## 6. Calibration and risk-coverage diagnostics

In [ ]:
primary = imputer.evaluations_[config.model.final_calibration_strategy]

plot_reliability(primary.reliability);
plot_risk_coverage(
    primary.risk_coverage,
    target_error=config.model.target_selective_error,
);
plot_confusion(primary.metrics);

## 7. Metrics by evidence regime

In [ ]:
for strategy, evaluation in imputer.evaluations_.items():
    print(f"\n{strategy.upper()}")
    display(evaluation.metrics_by_regime)

## 8. Inspect accepted errors and abstentions in cross-validation

Accepted high-confidence errors deserve manual review. They are usually more informative than another decimal place of aggregate accuracy.

In [ ]:
# Fast mode uses reconstruction. A final promotion run should add the harder
# blocked strategy and review its errors instead.
error_review_strategy = next(
    name
    for name in ("encounter", "blocked", "reconstruction", "purged_blocked")
    if name in imputer.evaluations_
)
print(f"Reviewing accepted errors from: {error_review_strategy}")
oof = imputer.evaluations_[error_review_strategy].oof_predictions.copy()
oof["IS_ERROR"] = oof["PREDICTED_CLASS"].ne(oof["MODEL_CLASS"])
oof["PREDICTED_CONFIDENCE"] = oof[["P_SRKW", "P_TRANSIENT"]].max(axis=1)

accepted_errors = oof.loc[oof["ACCEPTED"] & oof["IS_ERROR"]].sort_values(
    "PREDICTED_CONFIDENCE", ascending=False
)
abstention_counts = oof.loc[~oof["ACCEPTED"], "DECISION_REASON"].value_counts()

print(f"Accepted errors: {len(accepted_errors):,}")
display(accepted_errors)
display(abstention_counts.rename("N").to_frame())

In [ ]:
# 1. Corrected marine-network diagnostics
marine = predictions.loc[
    predictions["DISTANCE_METHOD"].eq("MARINE_LOOKUP")
].copy()

resolved = (
    marine["MARINE_ROUTED_NEIGHBORS"]
    + marine["MARINE_FALLBACK_NEIGHBORS"]
)

marine["LOOKUP_RATE_WHEN_RESOLVED"] = (
    marine["MARINE_ROUTED_NEIGHBORS"]
    .div(resolved.where(resolved.gt(0)))
)

marine["FALLBACK_RATE_WHEN_RESOLVED"] = (
    marine["MARINE_FALLBACK_NEIGHBORS"]
    .div(resolved.where(resolved.gt(0)))
)

marine["BARRIER_EXCLUSION_RATE"] = (
    marine["MARINE_BARRIER_EXCLUDED_NEIGHBORS"]
    .div(
        marine["MARINE_CANDIDATE_NEIGHBORS"].where(
            marine["MARINE_CANDIDATE_NEIGHBORS"].gt(0)
        )
    )
)

marine_summary = pd.Series(
    {
        "predictions": len(marine),
        "pct_with_routed_neighbor": 100 * resolved.gt(0).mean(),
        "lookup_rate_when_resolved": marine["LOOKUP_RATE_WHEN_RESOLVED"].mean(),
        "fallback_rate_when_resolved": marine["FALLBACK_RATE_WHEN_RESOLVED"].mean(),
        "mean_barrier_exclusion_rate": marine["BARRIER_EXCLUSION_RATE"].mean(),
        "median_barrier_exclusion_rate": marine["BARRIER_EXCLUSION_RATE"].median(),
        "pct_with_any_barrier_exclusion": (
            100 * marine["MARINE_BARRIER_EXCLUDED_NEIGHBORS"].gt(0).mean()
        ),
    },
    name="VALUE",
)

display(marine_summary.to_frame())
display(
    marine[
        [
            "MARINE_CANDIDATE_NEIGHBORS",
            "MARINE_ROUTED_NEIGHBORS",
            "MARINE_FALLBACK_NEIGHBORS",
            "MARINE_BARRIER_EXCLUDED_NEIGHBORS",
            "MARINE_OUT_OF_RANGE_NEIGHBORS",
            "BARRIER_EXCLUSION_RATE",
        ]
    ].quantile([0.50, 0.75, 0.90, 0.95, 0.99, 1.00])
)

In [ ]:
# 2. Predictions most affected by marine routing
review_columns = [
    "OBSERVATION_ID",
    "SIGHTING_DATE",
    "LATITUDE",
    "LONGITUDE",
    "DISPLAY_CLASS",
    "P_SRKW",
    "P_TRANSIENT",
    "IMPUTATION_APPLIED",
    "EVIDENCE_REGIME",
    "ABSTENTION_REASON",
    "NEAREST_SRKW_KM",
    "NEAREST_SRKW_DAY_LAG",
    "NEAREST_TRANSIENT_KM",
    "NEAREST_TRANSIENT_DAY_LAG",
    "MARINE_CANDIDATE_NEIGHBORS",
    "MARINE_ROUTED_NEIGHBORS",
    "MARINE_FALLBACK_NEIGHBORS",
    "MARINE_BARRIER_EXCLUDED_NEIGHBORS",
    "MARINE_OUT_OF_RANGE_NEIGHBORS",
    "MARINE_TARGET_SNAP_KM",
    "BARRIER_EXCLUSION_RATE",
]

top_barrier_cases = (
    marine.loc[marine["MARINE_CANDIDATE_NEIGHBORS"].ge(10), review_columns]
    .sort_values(
        ["BARRIER_EXCLUSION_RATE", "MARINE_BARRIER_EXCLUDED_NEIGHBORS"],
        ascending=False,
    )
    .head(50)
)

display(top_barrier_cases.head(20))

barrier_review_path = OUTPUT_DIR / "marine_barrier_review.csv"
top_barrier_cases.to_csv(barrier_review_path, index=False)
print(barrier_review_path)

In [ ]:
# 3. Accepted errors and calibration-bin sample sizes
error_frames = []
reliability_frames = []

for strategy in EVALUATION_STRATEGIES:
    evaluation = imputer.evaluations_[strategy]
    oof = evaluation.oof_predictions.copy()

    errors = oof.loc[
        oof["ACCEPTED"]
        & oof["PREDICTED_CLASS"].ne(oof["MODEL_CLASS"])
    ].copy()
    errors.insert(0, "STRATEGY", strategy)
    error_frames.append(errors)

    reliability = evaluation.reliability.copy()
    reliability.insert(0, "STRATEGY", strategy)
    reliability_frames.append(reliability)

accepted_error_review = pd.concat(error_frames, ignore_index=True)
reliability_review = pd.concat(reliability_frames, ignore_index=True)

display(
    accepted_error_review.groupby(
        ["STRATEGY", "MODEL_CLASS", "PREDICTED_CLASS"]
    ).size().rename("N").to_frame()
)

display(
    reliability_review[
        [
            "STRATEGY",
            "LOWER",
            "UPPER",
            "COUNT",
            "MEAN_P_SRKW",
            "OBSERVED_SRKW_RATE",
            "ABS_GAP",
        ]
    ]
)

errors_path = OUTPUT_DIR / "accepted_errors_review.csv"
reliability_path = OUTPUT_DIR / "reliability_bins_review.csv"

accepted_error_review.to_csv(errors_path, index=False)
reliability_review.to_csv(reliability_path, index=False)

print(errors_path)
print(reliability_path)

## 9. Inspect 2025 assignment counts

In [ ]:
predictions["SIGHTING_DATE"] = pd.to_datetime(predictions["SIGHTING_DATE"])
map_2025 = predictions.loc[predictions["SIGHTING_DATE"].dt.year.eq(2025)].copy()

class_counts_2025 = (
    map_2025["DISPLAY_CLASS"]
    .value_counts(dropna=False)
    .rename_axis("DISPLAY_CLASS")
    .rename("N")
    .to_frame()
)
display(class_counts_2025)

unknown_2025 = map_2025.loc[
    map_2025["DISPLAY_CLASS"].isin(
        ["SRKW_ASSIGNED", "TRANSIENT_ASSIGNED", "UNKNOWN_STILL"]
    )
].sort_values(["SIGHTING_DATE", "P_SRKW"], ascending=[True, False])

display(
    unknown_2025[
        [
            "OBSERVATION_ID",
            "SIGHTING_DATE",
            "LATITUDE",
            "LONGITUDE",
            "DISPLAY_CLASS",
            "P_SRKW",
            "P_TRANSIENT",
            "EVIDENCE_REGIME",
            "ABSTENTION_REASON",
        ]
    ].head(100)
)

## 10. Animated 2025 map

In [ ]:
map_figure = make_animated_map(
    predictions,
    year=2025,
    frame_unit="week",
    include_known_other=False,
    title="Known and selectively assigned ecotypes, 2025",
)
map_figure.show()

The HTML map saved in the output directory is self-contained for Plotly interactions. OpenStreetMap tiles still require network access when the map is viewed.

## 11. Optional operational end-of-day model

This is a separately fitted model. It includes same-date and earlier-date context but omits all later-date features. It represents an **end-of-day** contract, not an immediate intraday prediction, because the model consumes the completed set of records for the target date.

Set `RUN_OPERATIONAL = True` to build it.

In [ ]:
RUN_OPERATIONAL = False

if RUN_OPERATIONAL:
    operational_config = replace(
        config,
        regime="operational_eod",
        map_frame="week",
    )
    operational_result = run_imputation_workflow(
        workspace_root=REPO_ROOT,
        observations_path=OBSERVATIONS_PATH,
        associations_path=ASSOCIATIONS_PATH,
        output_dir=REPO_ROOT / "outputs/ecotype_imputation_operational_eod",
        config=operational_config,
        make_map=True,
        evaluate_strategies=EVALUATION_STRATEGIES,
    )
    display(
        pd.DataFrame(
            {
                name: evaluation.metrics
                for name, evaluation in operational_result.imputer.evaluations_.items()
            }
        ).T[["BRIER", "ROC_AUC", "COVERAGE", "SELECTIVE_ACCURACY", "SELECTIVE_ERROR_UPPER"]]
    )

## 12. Promotion checklist

Before treating the assigned layer as production historical truth:

1. Review accepted errors from reconstruction, encounter-held-out, and blocked validation.
2. Review results by source, era, geography, and evidence regime.
3. Create an expert-adjudicated sample of genuinely unknown historical records.
4. Lock the risk target before evaluating that adjudicated sample.
5. Preserve observed labels, probabilities, abstention reasons, model version, and assignment status downstream.

The model is designed to abstain. Lower coverage can be the correct result when the data cannot support the requested error guarantee.

## 13. Week 32, 2025 ecotype probability surface (H3 resolution 8)

This final diagnostic evaluates a complete water-only H3 layer at the midpoint of ISO week 32 (2025-08-07). The fitted model supplies locally supported seed probabilities; a Gaussian heat kernel propagates those probabilities across the connected H3 water graph. Color represents `P(SRKW)` versus `P(TRANSIENT)`, while opacity preserves the strength and proximity of the underlying model evidence. Water components with no connection to model-supported evidence remain transparent.

The layer is an inferred probability surface, not a set of synthetic sightings and not additional training data.

In [ ]:
import os

from marine_mammal_toolkit.tools.observations.impute import build_weekly_probability_surface
from marine_mammal_toolkit.tools.observations.impute import make_probability_surface_map

def read_repo_env(name: str) -> str | None:
    env_path = REPO_ROOT / "config/.env"
    if not env_path.exists():
        return None
    for raw_line in env_path.read_text().splitlines():
        line = raw_line.strip()
        if line and not line.startswith("#") and line.startswith(f"{name}="):
            return line.split("=", 1)[1].strip().strip("\"'") or None
    return None

MAPTILER_API_KEY = os.environ.get("MAPTILER_API_KEY") or read_repo_env("MAPTILER_API_KEY")
if MAPTILER_API_KEY is None:
    print("MAPTILER_API_KEY is not configured; using Carto Positron for this render.")
    print("Add MAPTILER_API_KEY=<your key> to config/.env and rerun this cell for MapTiler Landscape.")

surface_layer, week_start, week_end, surface_date = build_weekly_probability_surface(
    imputer,
    WATER_H3_GRID_PATH,
    year=2025,
    iso_week=43,
    surface_resolution=8,
    gaussian_sigma_km=8.0,
)
surface_figure = make_probability_surface_map(
    surface_layer,
    predictions,
    week_start=week_start,
    week_end=week_end,
    surface_date=surface_date,
    iso_week=43,
    year=2025,
    maptiler_api_key=MAPTILER_API_KEY,
    maptiler_style="landscape-v4",
)

surface_table_path = OUTPUT_DIR / "week_32_2025_ecotype_probability_surface_h3_r8.parquet"
surface_map_path = OUTPUT_DIR / "week_32_2025_ecotype_probability_surface_h3_r8.html"
surface_layer.to_parquet(surface_table_path, index=False)
surface_figure.write_html(surface_map_path, include_plotlyjs=True, full_html=True)
print(f"Complete connected-water surface cells: {len(surface_layer):,}")
print(f"Direct model seed cells: {surface_layer['IS_MODEL_SEED'].sum():,}")
print(f"Gaussian-interpolated cells: {(~surface_layer['IS_MODEL_SEED']).sum():,}")
print(f"Surface table written to: {surface_table_path}")
print(f"Surface map written to: {surface_map_path}")
surface_figure.show()

The surface uses the fitted model only for seed inference and then performs water-graph Gaussian interpolation. It is intentionally excluded from training and from the sightings count tables. Low-opacity cells are farther from direct model evidence; transparent cells belong to water components with no supported seed and are not evidence of absence. MapTiler requires a client API key, which can be provided as `MAPTILER_API_KEY` in the environment or `config/.env`.